# Eesti II samba fondide aktiivse juhtimise osakaal

Arvutame **aktiivse juhtimise osakaalu** (Active Share, Cremers & Petajisto 2009) Eesti II samba pensionifondidele. Kasutame andmetena Tuleva fondide röntgeni andmestikku mai-juuni 2026 seisuga. Andmestik on kättesaadav: https://github.com/StenEhrlich/estonian-pension-lookthrough

Iga fondi võrdlusindeksina kasutame sama fondivalitseja indeksfondist ning Tuleva Maailma Võlakirjade pensionifondi portfellidest loodud võrdlusindeksit.


In [1]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/StenEhrlich/estonian-pension-lookthrough/main"        # või "." — kohalik kaust

funds   = pd.read_csv(f"{BASE}/funds.csv", index_col="code")
sleeves = pd.read_csv(f"{BASE}/fund_sleeves.csv").pivot(index="code", columns="sleeve", values="weight_pct")
fond_h  = pd.read_csv(f"{BASE}/fund_holdings.csv")       # aktiivsete fondide positsioonid
bench_h = pd.read_csv(f"{BASE}/benchmark_holdings.csv")  # indeksfondid + TUK00


## 1. Valem

$$\text{Active Share} = \tfrac{1}{2}\sum_i \lvert w_{fond,i} - w_{indeks,i}\rvert$$

kus kaalud on protsentides ja kumbki pool summeerub 100-le. 0% = portfellid identsed, 100% = ühisosa puudub.


In [2]:
def vec(df, code, vector):
    """Ühe fondi positsioonid: kaal % väärtpaberi/emitendi kaupa (summa = 100)."""
    return df[(df.code == code) & (df.vector == vector)].set_index("key").weight_pct

def AS(fond, indeks):
    return 0.5 * fond.sub(indeks, fill_value=0).abs().sum()

# käsitsi kontroll: 60/40 vs 50/50 -> 0.5*(10+10) = 10
assert AS(pd.Series({"A": 60, "B": 40}), pd.Series({"A": 50, "B": 50})) == 10


## 2. Kolm mõõdikut

1. **Aktsiaosa AS** — fondi aktsiad (läbi vaadatud) vs fondivalitseja indeksfondi aktsiad. Puhas aktsiavalik.
2. **Võlakirjaosa AS** — fondi võlakirjad **emitendi** tasandil vs Tuleva Võlakirjad (TUK00). (SEB analüüs kasutas TUK00 nimepõhist vektorit — veerg `tuk00_variant` failis `funds.csv`.)
3. **Komposiit-AS** — kogu fond vs fondi **enda varajaotusega** kaalutud võrdlusindeks: aktsiaosa → indeksfond, muu → TUK00. Erakapital, kinnisvara ja kuld puuduvad võrdlusindeksist → jäävad aktiivseks; raha on neutraalne. SEB metoodikaerand (vt `*_method_spec.md`): indeksfond katab ka PE+kinnisvara+raha ning TUK00 täpselt võlakirjaosa.

Fondivektor ja võrdlusvektor ehitatakse ühesuguselt: iga sleeve'i vektor korrutatakse tema osakaaluga; jaotus varaklasside kaupa tuleb võtmeprefiksist (`EQ:` / `BD:` / PE / RE / KULD / RAHA).


In [3]:
def osa(v, kaal, prefiks=""):
    """Sleeve'i vektor (summa 100) -> osa kogu portfellist."""
    return (v * kaal / 100).rename(lambda k: prefiks + k)

def fondi_vektor(code):
    s = sleeves.loc[code] * 100 / sleeves.loc[code, "total"]
    return pd.concat([osa(vec(fond_h, code, "equity"), s["eq"], "EQ:"),
                      osa(vec(fond_h, code, "bond"), s["bond"], "BD:"),
                      pd.Series({"PE": s["pe"], "RE": s["re"], "KULD": s["gold"], "RAHA": s["cash"]})])

def vordlus_vektor(code):
    s = sleeves.loc[code] * 100 / sleeves.loc[code, "total"]
    indeks = vec(bench_h, funds.loc[code, "benchmark_code"], "equity")
    tuk00  = vec(bench_h, "TUK00", funds.loc[code, "tuk00_variant"])
    if funds.loc[code, "manager"] == "SEB":   # SEB erand: indeksfond katab ka PE+RE+raha
        aktsia, vlk, raha = s["eq"] + s["pe"] + s["re"] + s["cash"], s["bond"], 0.0
    else:                                     # teised: PE+RE+kuld -> TUK00, raha neutraalne
        aktsia, vlk, raha = s["eq"], s["bond"] + s["pe"] + s["re"] + s["gold"], s["cash"]
    return pd.concat([osa(indeks, aktsia, "EQ:"), osa(tuk00, vlk, "BD:"),
                      pd.Series({"RAHA": raha})])

def jaotus(F, B):
    """Komposiit-AS panus varaklasside kaupa (pp); summa = komposiit-AS."""
    vahe = 0.5 * F.sub(B, fill_value=0).abs()
    return vahe.groupby(vahe.index.str.split(":").str[0]).sum()


## 3. Arvutus — kõik fondid


In [4]:
read = []
for code, f in funds[funds.role == "active"].iterrows():
    aktsiad, vlk = vec(fond_h, code, "equity"), vec(fond_h, code, "bond")
    jag = jaotus(fondi_vektor(code), vordlus_vektor(code))
    read.append({"kood": code, "fond": f["name"],
        "aktsia_AS": AS(aktsiad, vec(bench_h, f.benchmark_code, "equity")) if len(aktsiad) else None,
        "vlk_AS": AS(vlk, vec(bench_h, "TUK00", f.tuk00_variant)) if len(vlk) else None,
        "komposiit_AS": jag.sum(), **jag.add_prefix("pp_")})
res = pd.DataFrame(read).set_index("kood").round(2)
res[["fond", "aktsia_AS", "vlk_AS", "komposiit_AS"]]


,fond,aktsia_AS,vlk_AS,komposiit_AS
kood,,,,
K1960,Swedbank Pensionifond K1960,50.59,75.48,68.54
K1970,Swedbank Pensionifond K1970,48.00,100.00,52.99
K1980,Swedbank Pensionifond K1980,47.97,100.00,51.60
K1990,Swedbank Pensionifond K1990,1.31,NaN,1.31
K2000,Swedbank Pensionifond K2000,49.98,NaN,49.96
KKONS,Swedbank Pensionifond Konservatiivne,NaN,76.92,76.31
LLK50,LHV Pensionifond Ettevõtlik,95.37,80.27,89.96
LMK25,LHV Pensionifond Tasakaalukas,99.36,80.27,80.53
LXK00,LHV Pensionifond Rahulik,100.00,80.27,76.87


## 4. Kontroll avaldatud tulemuste vastu

`reference_results.json` sisaldab töö avaldatud (sõltumatult üle vaadatud) väärtusi — iga fond peab klappima 0,01 pp täpsusega.


In [5]:
ref = pd.read_json(f"{BASE}/reference_results.json").T
for meie, avaldatud in [("aktsia_AS", "equity_AS"), ("vlk_AS", "bond_AS"), ("komposiit_AS", "composite_AS")]:
    erinevus = (res[meie] - ref[avaldatud].astype(float)).abs()
    assert erinevus.max() < 0.01, (meie, erinevus.idxmax())
print(f"kõik {len(res)} fondi klapivad avaldatud tulemustega")


kõik 18 fondi klapivad avaldatud tulemustega


## 5. Väljund töö jaoks

Komposiit-AS on töö ESMA tabeli (`tab:esma`) AS-veerg; jaotus (pp varaklassi kaupa) toetab tulemuste arutelu.


In [6]:
jrk = ["SEK100", "SEK50", "SEK25", "SEK00", "LXK75", "LLK50", "LMK25", "LXK00",
       "NPK75", "NPK50", "NPK25", "NPK00", "K1960", "K1970", "K1980", "K1990", "K2000", "KKONS"]
tabel = res.loc[jrk]
tabel.to_csv("active_share_results.csv")
for kood, r in tabel.iterrows():                    # LaTeX-read tab:esma jaoks
    print(f"{r['fond']:35s} & {r['komposiit_AS']:.1f} \\\\".replace(".", ","))
tabel


SEB Pensionifond 18+                & 38,8 \\
SEB Pensionifond 55+                & 42,5 \\
SEB Pensionifond 60+                & 53,1 \\
SEB Pensionifond 65+                & 55,9 \\
LHV Pensionifond Julge              & 84,2 \\
LHV Pensionifond Ettevõtlik         & 90,0 \\
LHV Pensionifond Tasakaalukas       & 80,5 \\
LHV Pensionifond Rahulik            & 76,9 \\
Luminor 16-50                       & 18,4 \\
Luminor 50-56                       & 29,7 \\
Luminor 56+                         & 49,0 \\
Luminor 61-65                       & 65,0 \\
Swedbank Pensionifond K1960         & 68,5 \\
Swedbank Pensionifond K1970         & 53,0 \\
Swedbank Pensionifond K1980         & 51,6 \\
Swedbank Pensionifond K1990         & 1,3 \\
Swedbank Pensionifond K2000         & 50,0 \\
Swedbank Pensionifond Konservatiivne & 76,3 \\


,fond,aktsia_AS,vlk_AS,komposiit_AS,pp_BD,pp_EQ,pp_KULD,pp_PE,pp_RAHA,pp_RE
kood,,,,,,,,,,
SEK100,SEB Pensionifond 18+,34.46,99.99,38.81,3.30,32.55,0.00,1.21,0.0,1.74
SEK50,SEB Pensionifond 55+,34.45,81.50,42.54,6.86,30.55,0.00,0.96,0.0,4.17
SEK25,SEB Pensionifond 60+,18.39,89.77,53.10,36.81,11.10,0.00,0.23,0.0,4.97
SEK00,SEB Pensionifond 65+,27.94,58.59,55.87,53.39,2.48,0.00,0.00,0.0,0.00
LXK75,LHV Pensionifond Julge,94.88,80.27,84.18,24.69,44.17,2.80,8.35,0.0,4.17
LLK50,LHV Pensionifond Ettevõtlik,95.37,80.27,89.96,33.00,30.08,2.87,16.81,0.0,7.20
LMK25,LHV Pensionifond Tasakaalukas,99.36,80.27,80.53,56.37,8.19,3.21,3.56,0.0,9.20
LXK00,LHV Pensionifond Rahulik,100.00,80.27,76.87,73.55,0.49,2.82,0.00,0.0,0.00
NPK75,Luminor 16-50,14.25,98.18,18.39,4.39,13.44,0.00,0.34,0.0,0.22
